# Layer Skip Analysis

What happens when individual layers are bypassed: logit impact,
residual contribution, redundancy, and cumulative effects.

## Why This Matters

Layer skip characterizes what each layer contributes to the model's computation. Understanding layer-level organization — which layers are critical, which are redundant, and how they specialize — is essential for both interpretability and efficient model design.

**Key references:**
- [Elhage et al. (2021) "A Mathematical Framework for Transformer Circuits"](https://transformer-circuits.pub/2021/framework/index.html) — Foundational framework for attention head and residual stream analysis

In [ ]:
import jax
import jax.numpy as jnp
from irtk import HookedTransformer, HookedTransformerConfig
from irtk.layer_skip_analysis import (
    layer_skip_logit_impact, layer_residual_contribution,
    layer_redundancy_analysis, layer_cumulative_effect,
    layer_skip_summary,
)

cfg = HookedTransformerConfig(
    n_layers=4, d_model=32, n_ctx=64, d_head=8, n_heads=4, d_vocab=100,
)
model = HookedTransformer(cfg)
key = jax.random.PRNGKey(42)
leaves, treedef = jax.tree.flatten(model)
new_leaves = []
for leaf in leaves:
    if isinstance(leaf, jnp.ndarray) and leaf.dtype == jnp.float32:
        key, subkey = jax.random.split(key)
        new_leaves.append(jax.random.normal(subkey, leaf.shape) * 0.1)
    else:
        new_leaves.append(leaf)
model = jax.tree.unflatten(treedef, new_leaves)
tokens = jnp.array([1, 15, 30, 45, 60, 75, 90])
print('Model ready')

## Layer Skip Logit Impact

How does skipping each layer affect the output?

In [ ]:
result = layer_skip_logit_impact(model, tokens, position=-1)
print(f"Full prediction: token {result['full_prediction']}")
print(f"Most critical layer: {result['most_critical_layer']}\n")
for p in result['per_layer']:
    flag = ' ← CHANGES PRED' if p['prediction_changes'] else ''
    bar = '█' * int(p['mse_logit_change'] * 5)
    print(f"  Layer {p['layer']}: MSE={p['mse_logit_change']:.4f} {bar}{flag}")

## Layer Residual Contribution

How much does each layer add to the residual stream?

In [ ]:
result = layer_residual_contribution(model, tokens)
print(f"Max contribution at layer {result['max_contribution_layer']}\n")
for p in result['per_layer']:
    bar = '█' * int(p['relative_contribution'] * 20)
    print(f"  Layer {p['layer']}: contrib={p['contribution_norm']:.4f}, "
          f"resid={p['residual_norm']:.4f}, ratio={p['relative_contribution']:.4f} {bar}")

## Layer Redundancy

Are adjacent layers producing similar outputs?

In [ ]:
result = layer_redundancy_analysis(model, tokens)
print(f"Redundant pairs: {result['n_redundant_pairs']}\n")
for p in result['pairs']:
    flag = ' [REDUNDANT]' if p['is_redundant'] else ''
    print(f"  Layers {p['layers']}: sim={p['similarity']:.4f}{flag}")

## Cumulative Effect

How does the prediction build up layer by layer?

In [ ]:
result = layer_cumulative_effect(model, tokens, position=-1)
print(f"Final: token {result['final_prediction']}, {result['n_changes']} changes\n")
for p in result['per_layer']:
    flag = ' ← CHANGED' if p['changed'] else ''
    print(f"  Layer {p['layer']}: pred={p['prediction']:3d}, "
          f"logit={p['max_logit']:.4f}, H={p['entropy']:.4f}{flag}")

## Layer Skip Summary

Combined overview of layer importance.

In [ ]:
result = layer_skip_summary(model, tokens, position=-1)
print(f"Most critical layer: {result['most_critical_layer']}\n")
for p in result['per_layer']:
    flag = ' [CRITICAL]' if p['prediction_changes_on_skip'] else ''
    print(f"  Layer {p['layer']}: MSE={p['mse_impact']:.4f}, "
          f"contrib={p['contribution_norm']:.4f}, "
          f"ratio={p['relative_contribution']:.4f}{flag}")